# GIS Phase 2 — Distance to MV grid + Distance bands

**Study area:** 21 municipalities of Norte Amazónica (clusters C1–C5)  
**Sources:**
- `comunidades_electricidad_2012.json` — community census points (electricity, 2012)
- `Centrales_Generadoras.json` — generation plants 2023
- `Líneas_de_Media_Tensión_2023.json` — MV lines 2023 *(see data issue note below)*

**Tasks:**
1. Distance of unelectrified communities to nearest MV grid infrastructure
2. Scale 2012 spatial distribution to 2024 unelectrified HH (Source C)
3. Aggregate by cluster × distance band
4. Generation inventory

**CRS for distance computation:** EPSG:32719 (UTM Zone 19S) — appropriate for northern Bolivia (~-69° to -64° longitude)

In [1]:
import os
import json
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from shapely.ops import nearest_points
import warnings
warnings.filterwarnings("ignore")

print(f"geopandas {gpd.__version__}")

geopandas 1.1.3


In [2]:
# ============================================================
# Configuration
# ============================================================

BASE_DIR   = Path(".")
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCE_C_PATH = Path("../exctraction of data/output/municipalities_counts.csv")

# Municipality → Cluster mapping  (cluster labels C1–C5)
CLUSTER_MAP = {
    "Exaltación":             "C1",
    "Ixiamas":                "C1",
    "Reyes":                  "C1",
    "Santa_Rosa_Beni":        "C1",
    "Bolpebra":               "C2",
    "Guayaramerín":           "C3",
    "Puerto_Gonzalo_Moreno":  "C3",
    "Riberalta":              "C3",
    "Bella_Flor":             "C4",
    "Filadelfia":             "C4",
    "Ingavi":                 "C4",
    "Nueva_Esperanza":        "C4",
    "Porvenir":               "C4",
    "Puerto_Rico":            "C4",
    "San_Lorenzo":            "C4",
    "San_Pedro":              "C4",
    "Santa_Rosa_Pando":       "C4",
    "Santos_Mercado":         "C4",
    "Sena":                   "C4",
    "Villa_Nueva":            "C4",
    "Cobija":                 "C5",
}

BAND_BINS   = [0, 5, 10, 25, 50, np.inf]
BAND_LABELS = ["<5 km", "5-10 km", "10-25 km", "25-50 km", ">50 km"]

UTM_CRS = "EPSG:32719"  # UTM Zone 19S, central meridian -69°

# Bounding box for region (with buffer) to filter Centrales
REGION_BBOX = dict(min_lon=-70.0, max_lon=-63.5, min_lat=-13.0, max_lat=-9.5)

print("Configuration loaded.")
print(f"Municipalities: {len(CLUSTER_MAP)}")
print(f"Clusters: {sorted(set(CLUSTER_MAP.values()))}")

Configuration loaded.
Municipalities: 21
Clusters: ['C1', 'C2', 'C3', 'C4', 'C5']


In [3]:
# ============================================================
# Load the 3 JSON files
# ============================================================

print("Loading comunidades_electricidad_2012.json  (large file, ~19k features) ...")
com_gdf = gpd.read_file(DATA_DIR / "comunidades_electricidad_2012.json")
com_gdf.set_crs("EPSG:4326", inplace=True, allow_override=True)
print(f"  {len(com_gdf)} community features loaded")
print(f"  Columns: {list(com_gdf.columns)}")

print("\nLoading Centrales_Generadoras.json ...")
gen_gdf = gpd.read_file(DATA_DIR / "Centrales_Generadoras.json")
gen_gdf.set_crs("EPSG:4326", inplace=True, allow_override=True)
print(f"  {len(gen_gdf)} generation plant features loaded")

print("\nLoading Líneas_de_Media_Tensión_2023.json ...")
lineas_gdf = gpd.read_file(DATA_DIR / "Líneas_de_Media_Tensión_2023.json")
lineas_gdf.set_crs("EPSG:4326", inplace=True, allow_override=True)
print(f"  {len(lineas_gdf)} features loaded from Líneas file")

print("\nLoading Source C (2024 unelectrified HH) ...")
source_c = pd.read_csv(SOURCE_C_PATH)
source_c = source_c[source_c["municipality"] != "TOTAL"].copy()
print(f"  {len(source_c)} municipalities, total = {source_c['non_elec_hh'].sum()} unelec. HH")
print(source_c[["municipality", "non_elec_hh"]].to_string(index=False))

Loading comunidades_electricidad_2012.json  (large file, ~19k features) ...


  19279 community features loaded
  Columns: ['id', 'fid', 'OBJECTID', 'gid', 'codine', 'poblacion', 'vivpartic', 'Depto', 'Provincia', 'Municipio', 'Comunidad', 'Area', 'Hogares', 'Red_elec', 'Motor', 'Panel', 'Otra', 'Hog_Elec', 'No_tiene', 'Cobertura', 'geometry']

Loading Centrales_Generadoras.json ...
  48 generation plant features loaded

Loading Líneas_de_Media_Tensión_2023.json ...
  48 features loaded from Líneas file

Loading Source C (2024 unelectrified HH) ...
  21 municipalities, total = 11389 unelec. HH
         municipality  non_elec_hh
           Exaltación          273
         Guayaramerín          825
                Reyes          711
            Riberalta         2925
      Santa_Rosa_Beni          407
              Ixiamas         1043
           Bella_Flor          351
             Bolpebra          253
               Cobija          395
           Filadelfia          594
               Ingavi          261
      Nueva_Esperanza          151
             Porvenir 

In [4]:
# ============================================================
# DATA QUALITY CHECK — Líneas JSON
# ============================================================

lineas_geom_types = lineas_gdf.geometry.geom_type.unique()
gen_cols          = set(gen_gdf.columns) - {"geometry"}
lineas_cols       = set(lineas_gdf.columns) - {"geometry"}

print("=" * 65)
print("DATA QUALITY CHECK — Líneas_de_Media_Tensión_2023.json")
print("=" * 65)
print(f"Expected geometry  : MultiLineString or LineString")
print(f"Actual geometry    : {lineas_geom_types}")
print(f"Lineas columns     : {sorted(lineas_cols)}")
print(f"Centrales columns  : {sorted(gen_cols)}")
print(f"Lineas features    : {len(lineas_gdf)}")
print(f"Centrales features : {len(gen_gdf)}")
print()

if lineas_cols == gen_cols and len(lineas_gdf) == len(gen_gdf):
    print("WARNING: Líneas_de_Media_Tensión_2023.json is a duplicate of")
    print("         Centrales_Generadoras.json (same columns, same feature count).")
    print("         MV line geometry (LineString) is MISSING from the JSON file.")
    print("         Likely cause: wrong layer exported from GIS.")
    print()
    print("WORKAROUND used in this notebook:")
    print("  - Distance to nearest GENERATION PLANT is used as proxy for")
    print("    distance to the MV grid.")
    print("  - Generation plants are typically located on or adjacent to the")
    print("    MV grid, so this provides a conservative lower-bound estimate.")
    print()
    print("ACTION NEEDED: Re-export Líneas_de_Media_Tensión_2023 from QGIS/ArcGIS")
    print("  (File > Save Layer As > GeoJSON) then re-run this notebook.")
    USE_PROXY = True
else:
    print("Líneas file appears to contain MV line geometry — using real distances.")
    USE_PROXY = False

print("=" * 65)

DATA QUALITY CHECK — Líneas_de_Media_Tensión_2023.json
Expected geometry  : MultiLineString or LineString
Actual geometry    : ['Point']
Lineas columns     : ['Central', 'Pot_kW', 'Propietari', 'Tipo', 'fid', 'fid_1', 'id']
Centrales columns  : ['Central', 'Pot_kW', 'Propietari', 'Tipo', 'fid', 'fid_1', 'id']
Lineas features    : 48
Centrales features : 48

         Centrales_Generadoras.json (same columns, same feature count).
         MV line geometry (LineString) is MISSING from the JSON file.
         Likely cause: wrong layer exported from GIS.

WORKAROUND used in this notebook:
  - Distance to nearest GENERATION PLANT is used as proxy for
    distance to the MV grid.
  - Generation plants are typically located on or adjacent to the
    MV grid, so this provides a conservative lower-bound estimate.

ACTION NEEDED: Re-export Líneas_de_Media_Tensión_2023 from QGIS/ArcGIS
  (File > Save Layer As > GeoJSON) then re-run this notebook.


In [5]:
# ============================================================
# Region filter — keep only communities in the 21 study municipalities
# ============================================================

# Matching rules: (standard_name, dept_substring, muni_substring)
# Uses str.contains (case-insensitive) on both Depto and Municipio columns
MUNI_FILTER = [
    ("Exaltación",            "Beni",   "Exaltac"),
    ("Guayaramerín",          "Beni",   "Guayaramer"),    # encoding-safe prefix
    ("Reyes",                 "Beni",   "Reyes"),
    ("Riberalta",             "Beni",   "Riberalta"),
    ("Santa_Rosa_Beni",       "Beni",   "Santa Rosa"),    # 'Tercera Sección - Santa Rosa'
    ("Ixiamas",               "La Paz", "Ixiamas"),
    ("Bella_Flor",            "Pando",  "Bella Flor"),
    ("Bolpebra",              "Pando",  "Bolpebra"),
    ("Cobija",                "Pando",  "Cobija"),
    ("Filadelfia",            "Pando",  "Filadelfia"),
    ("Ingavi",                "Pando",  "Ingavi"),
    ("Nueva_Esperanza",       "Pando",  "Nueva Esperanza"),
    ("Porvenir",              "Pando",  "Porvenir"),
    ("Puerto_Gonzalo_Moreno", "Pando",  "Puerto Gonzalo"),
    ("Puerto_Rico",           "Pando",  "Puerto Rico"),
    ("San_Lorenzo",           "Pando",  "San Lorenzo"),
    ("San_Pedro",             "Pando",  "San Pedro"),
    ("Santa_Rosa_Pando",      "Pando",  "Santa Rosa del Abun"),  # 'Santa Rosa del Abuná'
    ("Santos_Mercado",        "Pando",  "Santos Mercado"),
    ("Sena",                  "Pando",  "Sena"),
    ("Villa_Nueva",           "Pando",  "Villa Nueva"),
]

parts = []
for std_name, dept, muni_sub in MUNI_FILTER:
    mask = (
        com_gdf["Depto"].str.contains(dept, case=False, na=False) &
        com_gdf["Municipio"].str.contains(muni_sub, case=False, na=False)
    )
    sub = com_gdf[mask].copy()
    sub["municipio_std"] = std_name
    sub["cluster"]       = CLUSTER_MAP[std_name]
    parts.append(sub)

region_gdf = pd.concat(parts, ignore_index=True)
region_gdf["No_tiene"]  = pd.to_numeric(region_gdf["No_tiene"],  errors="coerce").fillna(0)
region_gdf["poblacion"] = pd.to_numeric(region_gdf["poblacion"], errors="coerce").fillna(0)
region_gdf["Hogares"]   = pd.to_numeric(region_gdf["Hogares"],   errors="coerce").fillna(0)

print(f"Study area: {len(region_gdf)} communities across {region_gdf['municipio_std'].nunique()} municipalities")
print(f"Communities with No_tiene > 0: {(region_gdf['No_tiene'] > 0).sum()}")

summary_region = region_gdf.groupby("municipio_std").agg(
    N_communities=("municipio_std", "count"),
    Sum_No_tiene=("No_tiene", "sum"),
    Sum_Hogares=("Hogares", "sum"),
    cluster=("cluster", "first"),
).sort_values("cluster")
print("\n" + summary_region.to_string())

Study area: 727 communities across 21 municipalities
Communities with No_tiene > 0: 697

                       N_communities  Sum_No_tiene  Sum_Hogares cluster
municipio_std                                                          
Exaltación                        24         271.0        938.0      C1
Santa_Rosa_Beni                   29         311.0       1976.0      C1
Ixiamas                           79         986.0       2270.0      C1
Reyes                             48         963.0       3018.0      C1
Bolpebra                          16         306.0        548.0      C2
Puerto_Gonzalo_Moreno             21         370.0       1026.0      C3
Guayaramerín                      49         961.0       9260.0      C3
Riberalta                         94        2562.0      19588.0      C3
Santos_Mercado                    15         200.0        394.0      C4
Santa_Rosa_Pando                  15         220.0        652.0      C4
San_Pedro                         17         42

In [6]:
# ============================================================
# Task 1 — Distance to nearest MV grid infrastructure
# ============================================================
# CRS: EPSG:32719 (UTM Zone 19S) — metric, appropriate for northern Bolivia

print(f"Reprojecting layers to {UTM_CRS} ...")
region_utm = region_gdf.to_crs(UTM_CRS)
gen_utm    = gen_gdf.to_crs(UTM_CRS)

if not USE_PROXY:
    lineas_utm = lineas_gdf.to_crs(UTM_CRS)

# Keep communities with No_tiene > 0
unelec = region_utm[region_utm["No_tiene"] > 0].copy()
print(f"Unelectrified communities (No_tiene > 0): {len(unelec)}")

# Build reference geometry for distance computation
if USE_PROXY:
    # Filter generation plants to region bounding box
    bbox = REGION_BBOX
    gen_region = gen_utm[
        (gen_utm.geometry.x >= -1e10) |  # utm x always valid
        (gen_utm.geometry.y >= -1e10)
    ].copy()
    # Filter using original WGS84 bounds
    gen_wgs84 = gen_gdf[
        (gen_gdf.geometry.x >= bbox["min_lon"]) &
        (gen_gdf.geometry.x <= bbox["max_lon"]) &
        (gen_gdf.geometry.y >= bbox["min_lat"]) &
        (gen_gdf.geometry.y <= bbox["max_lat"])
    ].copy()
    gen_region = gen_wgs84.to_crs(UTM_CRS)
    ref_union  = gen_region.geometry.unary_union
    distance_label = "dist_to_nearest_gen_plant_km  [PROXY — real MV line geometry missing]"
    print(f"Using {len(gen_region)} generation plants as MV grid proxy.")
else:
    ref_union = lineas_utm.geometry.unary_union
    distance_label = "dist_to_nearest_MV_line_km"
    print(f"Using {len(lineas_gdf)} MV line segments.")

print("Computing distances (may take a few seconds) ...")

def dist_to_ref(geom):
    _, nearest = nearest_points(geom, ref_union)
    return geom.distance(nearest) / 1000  # m → km

unelec["distance_km"] = unelec.geometry.apply(dist_to_ref)

# Distance band classification
unelec["dist_band"] = pd.cut(
    unelec["distance_km"],
    bins=BAND_BINS,
    labels=BAND_LABELS,
    right=True,
    include_lowest=True,
)

print(f"\nDistance computation complete.")
print(f"Column label: {distance_label}")
print(f"\nDistance range: {unelec['distance_km'].min():.1f} – {unelec['distance_km'].max():.1f} km")
print(f"Median distance: {unelec['distance_km'].median():.1f} km")

print("\nUnelectrified communities by distance band (2012 No_tiene HH):")
band_summary = unelec.groupby("dist_band", observed=True).agg(
    n_communities=("municipio_std", "count"),
    hh_no_tiene=("No_tiene", "sum"),
).assign(share=lambda df: (df["hh_no_tiene"] / df["hh_no_tiene"].sum()).round(3))
print(band_summary.to_string())

Reprojecting layers to EPSG:32719 ...
Unelectrified communities (No_tiene > 0): 697
Using 22 generation plants as MV grid proxy.
Computing distances (may take a few seconds) ...

Distance computation complete.
Column label: dist_to_nearest_gen_plant_km  [PROXY — real MV line geometry missing]

Distance range: 0.0 – 279.1 km
Median distance: 47.1 km

Unelectrified communities by distance band (2012 No_tiene HH):
           n_communities  hh_no_tiene  share
dist_band                                   
<5 km                 34       2854.0  0.217
5-10 km               47        727.0  0.055
10-25 km             127       2272.0  0.173
25-50 km             161       2289.0  0.174
>50 km               328       4990.0  0.380


In [ ]:
# Save Task 1 output
# When USE_PROXY=False (real MV line geometry), save to a new file so the
# plant-proxy CSV (task1_community_distances.csv) is never overwritten.
task1_out = unelec[[
    "municipio_std", "cluster", "Municipio", "Comunidad",
    "No_tiene", "poblacion", "distance_km", "dist_band",
]].copy()
task1_out.columns = [
    "Municipio_std", "Cluster", "Municipio_raw", "Comunidad",
    "No_tiene_2012", "Poblacion_2012", "distance_km", "Dist_band",
]
task1_out = task1_out.sort_values(["Cluster", "Municipio_std", "distance_km"]).reset_index(drop=True)

out_path = OUTPUT_DIR / ("task1_community_distances.csv" if USE_PROXY
                         else "task1_community_distances_lines.csv")
task1_out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved: {out_path}")
print(f"Shape: {task1_out.shape}")
print("\nSample (5 rows):")
print(task1_out.head(5).to_string(index=False))

In [8]:
# ============================================================
# Task 2 — Scale 2012 geography to 2024 quantities (Source C)
# ============================================================
# Assumption: 2012 spatial distribution of No_tiene is assumed to hold in 2024.
# Each community receives:
#   hh_2024_i = HH_2024_muni × (No_tiene_2012_i / Σ No_tiene_2012_in_muni)

print("ASSUMPTION: 2012 community-level distribution of unelectrified HH")
print("(No_tiene) is used as a spatial weight to allocate each municipality's")
print("2024 Source C count across its community points.")
print()

# Source C lookup dict: municipality_std → 2024 non_elec_hh
source_c_dict = source_c.set_index("municipality")["non_elec_hh"].to_dict()

# Map Source C to unelec communities
unelec["total_2024"] = unelec["municipio_std"].map(source_c_dict)

missing = unelec[unelec["total_2024"].isna()]["municipio_std"].unique()
if len(missing) > 0:
    print(f"WARNING: No Source C match found for: {missing}")
    print("  Check that municipality names match between CLUSTER_MAP and municipalities_counts.csv")
else:
    print("All 21 municipalities matched to Source C. OK.")

# Spatial weights within each municipality (normalized No_tiene)
muni_sum_2012 = unelec.groupby("municipio_std")["No_tiene"].transform("sum")
unelec["weight_2012"]    = unelec["No_tiene"] / muni_sum_2012.replace(0, np.nan)
unelec["hh_2024_scaled"] = unelec["weight_2012"] * unelec["total_2024"]

# Sanity check: sum per municipality should match Source C
check = unelec.groupby("municipio_std").agg(
    scaled_sum=("hh_2024_scaled", "sum"),
    source_c=("total_2024", "first"),
).round(1)
check["diff"] = (check["scaled_sum"] - check["source_c"]).abs()
print("\nScaling validation (diff should be ~0):")
print(check.to_string())

total_scaled = unelec["hh_2024_scaled"].sum()
total_source_c = source_c["non_elec_hh"].sum()
print(f"\nTotal 2024 HH (scaled): {total_scaled:.1f}")
print(f"Total Source C:          {total_source_c}")

ASSUMPTION: 2012 community-level distribution of unelectrified HH
(No_tiene) is used as a spatial weight to allocate each municipality's
2024 Source C count across its community points.

All 21 municipalities matched to Source C. OK.

Scaling validation (diff should be ~0):
                       scaled_sum  source_c  diff
municipio_std                                    
Bella_Flor                  351.0       351   0.0
Bolpebra                    253.0       253   0.0
Cobija                      395.0       395   0.0
Exaltación                  273.0       273   0.0
Filadelfia                  594.0       594   0.0
Guayaramerín                825.0       825   0.0
Ingavi                      261.0       261   0.0
Ixiamas                    1043.0      1043   0.0
Nueva_Esperanza             151.0       151   0.0
Porvenir                    213.0       213   0.0
Puerto_Gonzalo_Moreno       348.0       348   0.0
Puerto_Rico                 325.0       325   0.0
Reyes                    

In [9]:
# ============================================================
# Task 3 — Aggregate by cluster × distance band
# ============================================================

task3 = unelec.groupby(["cluster", "dist_band"], observed=True).agg(
    HH_2024_scaled   = ("hh_2024_scaled",  "sum"),
    N_communities    = ("municipio_std",    "count"),
    Mean_dist_km     = ("distance_km",      "mean"),
    Median_dist_km   = ("distance_km",      "median"),
).reset_index()

# Share of cluster total per band
cluster_totals = unelec.groupby("cluster")["hh_2024_scaled"].sum().rename("cluster_total_2024")
task3 = task3.merge(cluster_totals, on="cluster")
task3["Share_of_cluster"] = (task3["HH_2024_scaled"] / task3["cluster_total_2024"]).round(4)

# Round numeric columns
task3["HH_2024_scaled"]  = task3["HH_2024_scaled"].round(1)
task3["Mean_dist_km"]    = task3["Mean_dist_km"].round(2)
task3["Median_dist_km"]  = task3["Median_dist_km"].round(2)

task3 = task3.rename(columns={"dist_band": "Dist_band", "cluster": "Cluster"})
task3 = task3[["Cluster", "Dist_band", "HH_2024_scaled", "N_communities",
               "Mean_dist_km", "Median_dist_km", "Share_of_cluster"]]

print("=" * 75)
print("Task 3 — Cluster × Distance band table")
print("=" * 75)
print(task3.to_string(index=False))

print("\n--- Share of unelectrified HH beyond 50 km per cluster ---")
beyond50 = task3[task3["Dist_band"] == ">50 km"].set_index("Cluster")["Share_of_cluster"]
for cl in sorted(CLUSTER_MAP.values()):
    sh = beyond50.get(cl, 0.0)
    print(f"  {cl}: {sh*100:.1f}%")

out_path = OUTPUT_DIR / "task3_cluster_band.csv"
task3.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nSaved: {out_path}")

Task 3 — Cluster × Distance band table
Cluster Dist_band  HH_2024_scaled  N_communities  Mean_dist_km  Median_dist_km  Share_of_cluster
     C1     <5 km            68.5              4          1.27            0.30            0.0281
     C1   5-10 km            27.2              2          7.80            7.80            0.0112
     C1  10-25 km            10.1              2         13.71           13.71            0.0041
     C1  25-50 km           156.4             11         39.15           38.95            0.0643
     C1    >50 km          2171.8            145        165.44          143.87            0.8923
     C2  25-50 km           130.6              6         43.39           45.79            0.5163
     C2    >50 km           122.4             10         68.11           67.26            0.4837
     C3     <5 km          1512.1             15          2.46            2.56            0.3690
     C3   5-10 km           526.0             33          7.87            8.61          

In [10]:
# Cluster-level summary
cluster_summary = unelec.groupby("cluster").agg(
    HH_2024_total    = ("hh_2024_scaled", "sum"),
    N_communities    = ("municipio_std",   "count"),
    Mean_dist_km     = ("distance_km",     "mean"),
    Median_dist_km   = ("distance_km",     "median"),
    Max_dist_km      = ("distance_km",     "max"),
).round(2)
print("=" * 65)
print("Cluster-level summary")
print("=" * 65)
print(cluster_summary.to_string())

Cluster-level summary
         HH_2024_total  N_communities  Mean_dist_km  Median_dist_km  Max_dist_km
cluster                                                                         
C1              2434.0            164        149.19          129.14       279.07
C2               253.0             16         58.84           58.76        85.28
C3              4098.0            160         24.56           16.66        86.60
C4              4209.0            343         47.17           45.41       147.24
C5               395.0             14          8.24            6.44        22.57


In [11]:
# ============================================================
# Task 4 — Generation inventory for the study region
# ============================================================

bbox = REGION_BBOX
gen_region = gen_gdf[
    (gen_gdf.geometry.x >= bbox["min_lon"]) &
    (gen_gdf.geometry.x <= bbox["max_lon"]) &
    (gen_gdf.geometry.y >= bbox["min_lat"]) &
    (gen_gdf.geometry.y <= bbox["max_lat"])
].copy()

print(f"Generation plants in study region ({len(gen_region)} total):")
print(gen_region[["Central", "Tipo", "Pot_kW", "Propietari"]].to_string(index=False))

print("\n--- Inventory by Technology Type ---")
inventory = gen_region.groupby("Tipo").agg(
    N_plants     = ("Central", "count"),
    Total_kW     = ("Pot_kW",  "sum"),
).sort_values("Total_kW", ascending=False)
inventory["Share_%"] = (inventory["Total_kW"] / inventory["Total_kW"].sum() * 100).round(1)
print(inventory.to_string())
print(f"\nTotal installed capacity in region: {gen_region['Pot_kW'].sum():.0f} kW")

print("\n--- HYDRO flag ---")
hydro = gen_region[gen_region["Tipo"].str.contains("[Hh]idro|[Hh]ydro", na=False, regex=True)]
if len(hydro) > 0:
    print(f"HYDRO PLANTS FOUND ({len(hydro)}):")
    print(hydro[["Central", "Tipo", "Pot_kW", "Propietari"]].to_string(index=False))
    print("EnergyScope should INCLUDE a hydro technology for this region.")
else:
    print("No hydro plants in study region.")
    print("EnergyScope should NOT include hydro generation for Norte Amazónica.")

out_path = OUTPUT_DIR / "task4_generation_inventory.csv"
gen_region[["Central", "Tipo", "Pot_kW", "Propietari"]].sort_values(
    ["Tipo", "Pot_kW"], ascending=[True, False]
).to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nSaved: {out_path}")

Generation plants in study region (22 total):
                   Central                            Tipo  Pot_kW                Propietari
        Cachuela Esperanza Termoelectrica Diesel o Gas Oil   840.0             ENDE DELEBENI
                 Riberalta Termoelectrica Diesel o Gas Oil 31400.0                      ENDE
                  La Salud Termoelectrica Diesel o Gas Oil    80.0              ENDE DELBENI
                    Calama Termoelectrica Diesel o Gas Oil    24.0              ENDE DELBENI
                    Cayoba Termoelectrica Diesel o Gas Oil   180.0              ENDE DELBENI
          Rosario del Yata Termoelectrica Diesel o Gas Oil  1210.0              ENDE DELBENI
       Planta Solar Cobija               Solar Fotovotaica  5100.0           ENDE Guaracachi
              Puerto Siles Termoelectrica Diesel o Gas Oil    10.0              ENDE DELBENI
               Cooperativa Termoelectrica Diesel o Gas Oil    16.0              ENDE DELBENI
San Juan de Puerto Ustar

In [12]:
# ============================================================
# Final summary
# ============================================================

print("=" * 65)
print("FINAL SUMMARY — GIS Phase 2")
print("=" * 65)
print()
print(f"CRS used for distance:  {UTM_CRS} (UTM Zone 19S)")
print(f"Distance metric:        {'Proxy — distance to nearest generation plant' if USE_PROXY else 'Distance to nearest MV line'}")
print()
print(f"Communities in study area:     {len(region_gdf):>6}")
print(f"Unelectrified communities:     {len(unelec):>6}  (No_tiene > 0)")
print(f"Total 2024 unelec. HH (src C): {int(source_c['non_elec_hh'].sum()):>6}")
print(f"Total 2024 HH (scaled check):  {unelec['hh_2024_scaled'].sum():>9.1f}")
print()
print("Outputs:")
for f in OUTPUT_DIR.iterdir():
    print(f"  {f.name}")

if USE_PROXY:
    print()
    print("ACTION NEEDED: Provide correct Líneas_de_Media_Tensión_2023.json")
    print("  (with LineString geometry) to replace the proxy distances.")

FINAL SUMMARY — GIS Phase 2

CRS used for distance:  EPSG:32719 (UTM Zone 19S)
Distance metric:        Proxy — distance to nearest generation plant

Communities in study area:        727
Unelectrified communities:        697  (No_tiene > 0)
Total 2024 unelec. HH (src C):  11389
Total 2024 HH (scaled check):    11389.0

Outputs:
  task1_community_distances.csv
  task3_cluster_band.csv
  task4_generation_inventory.csv

ACTION NEEDED: Provide correct Líneas_de_Media_Tensión_2023.json
  (with LineString geometry) to replace the proxy distances.


## Notes

### Data issue — Líneas_de_Media_Tensión_2023.json
Both `Líneas_de_Media_Tensión_2023.json` and `Centrales_Generadoras.json` contain the same 48 Point features with fields `Propietari / Tipo / Central / Pot_kW`. The MV line geometry (MultiLineString) is missing. **To fix**: open your original GIS project, right-click the *Líneas de Media Tensión 2023* layer → Save As → GeoJSON → save to `analyse_GIS_phase2/data/Líneas_de_Media_Tensión_2023.json`, then re-run this notebook.

### Assumption — Task 2 scaling
The 2012 community-level distribution of `No_tiene` (unelectrified households from the INE census) is assumed to remain representative in 2024. Each community receives a proportional share of its municipality's 2024 Source C total.

### CRS
EPSG:32719 (UTM Zone 19S, central meridian −69°) was chosen as it centres the study area (−68° to −64° longitude). Maximum scale distortion at the eastern edge (~−64°) is <0.1%.

### Next steps
- Provide correct Líneas JSON for real MV-line distances
- Use `task1_community_distances.csv` to inspect the distance distribution
- Use `task3_cluster_band.csv` to set `share_dispersion` thresholds in EnergyScope inputs

In [ ]:
# ============================================================
# HARD GATE — verify Líneas_de_Media_Tensión_2023.json (re-downloaded)
# ============================================================
# Do not proceed if ANY check fails. Do NOT fall back to the plant proxy.

print("Loading Líneas_de_Media_Tensión_2023.json ...")
lineas_v2 = gpd.read_file(DATA_DIR / "Líneas_de_Media_Tensión_2023.json")
lineas_v2.set_crs("EPSG:4326", inplace=True, allow_override=True)

_geom_types = lineas_v2.geometry.geom_type.unique()
print(f"Feature count : {len(lineas_v2)}")
print(f"Geometry types: {_geom_types}")
print(f"Columns       : {sorted(set(lineas_v2.columns) - {'geometry'})}")

print("\nSIN field — distinct values and feature count per value:")
if "SIN" in lineas_v2.columns:
    print(lineas_v2["SIN"].value_counts().to_string())
else:
    print("  SIN field NOT FOUND")

with open(DATA_DIR / "Centrales_Generadoras.json", "rb") as _f:
    _gen_bytes = _f.read()
with open(DATA_DIR / "Líneas_de_Media_Tensión_2023.json", "rb") as _f:
    _ln_bytes = _f.read()
_byte_dup = (_ln_bytes == _gen_bytes)
print(f"\nByte-identical to Centrales : {_byte_dup}  "
      f"({len(_ln_bytes):,} B vs {len(_gen_bytes):,} B)")

_v_geom  = set(_geom_types).issubset({"LineString", "MultiLineString"})
_v_cnt   = len(lineas_v2) != 48
_v_sin   = "SIN" in lineas_v2.columns
_v_bytes = not _byte_dup

print("\nGATE CHECKS:")
print(f"  [{'OK' if _v_geom  else 'FAIL'}] Geometry is LineString/MultiLineString")
print(f"  [{'OK' if _v_cnt   else 'FAIL'}] Feature count ≠ 48  ({len(lineas_v2)} features)")
print(f"  [{'OK' if _v_sin   else 'FAIL'}] SIN field present")
print(f"  [{'OK' if _v_bytes else 'FAIL'}] Not byte-identical to Centrales_Generadoras.json")

if not (_v_geom and _v_cnt and _v_sin and _v_bytes):
    raise RuntimeError("HARD GATE FAILED — file is still corrupted. Do not proceed.")

print("\nALL GATE CHECKS PASSED — proceeding with real MV line distances.")

In [ ]:
# ============================================================
# Task 1 (lines) — Distance to nearest MV line per community
# ============================================================
# Four columns per community:
#   distance_km            — distance to nearest MV line (any SIN type)
#   nearest_line_type      — SIN value of that nearest line
#   distance_SIN_km        — distance to nearest SIN (interconnected) line
#   distance_isolated_km   — distance to nearest Sist. Ais. (isolated) line

print(f"Reprojecting {len(lineas_v2)} line features to {UTM_CRS} ...")
lineas_utm2     = lineas_v2.to_crs(UTM_CRS)
lineas_SIN      = lineas_utm2[lineas_utm2["SIN"] == "SIN"]
lineas_isolated = lineas_utm2[lineas_utm2["SIN"] == "Sist. Ais."]
print(f"  SIN lines      : {len(lineas_SIN)}")
print(f"  Isolated lines : {len(lineas_isolated)}")

print("Building unary_unions ...")
union_all_l = lineas_utm2.geometry.unary_union
union_SIN_l = lineas_SIN.geometry.unary_union
union_iso_l = lineas_isolated.geometry.unary_union

# Start from unelec (697 communities, already has hh_2024_scaled from Task 2).
# Drop existing distance columns — they depend on USE_PROXY and are not needed here.
unelec_lines = unelec.drop(
    columns=["distance_km", "dist_band", "weight_2012", "total_2024"], errors="ignore"
).copy()
assert len(unelec_lines) == 697, f"Expected 697 communities, got {len(unelec_lines)}"

print(f"Computing distances for {len(unelec_lines)} communities (may take ~1 min) ...")

import shapely as _shp
_pts = np.array(unelec_lines.geometry)

unelec_lines["distance_km"]          = _shp.distance(_pts, union_all_l) / 1000
unelec_lines["distance_SIN_km"]      = _shp.distance(_pts, union_SIN_l) / 1000
unelec_lines["distance_isolated_km"] = _shp.distance(_pts, union_iso_l) / 1000
unelec_lines["nearest_line_type"] = np.where(
    unelec_lines["distance_SIN_km"] <= unelec_lines["distance_isolated_km"],
    "SIN", "Sist. Ais."
)

unelec_lines["dist_band"] = pd.cut(
    unelec_lines["distance_km"],
    bins=BAND_BINS, labels=BAND_LABELS, right=True, include_lowest=True,
)

print("Done.")
print(f"\ndistance_km (nearest any MV line):")
print(f"  Range  : {unelec_lines['distance_km'].min():.2f} – {unelec_lines['distance_km'].max():.2f} km")
print(f"  Median : {unelec_lines['distance_km'].median():.2f} km")
print(f"  Mean   : {unelec_lines['distance_km'].mean():.2f} km")
print(f"\ndistance_SIN_km      : {unelec_lines['distance_SIN_km'].min():.2f}–{unelec_lines['distance_SIN_km'].max():.2f} km")
print(f"distance_isolated_km : {unelec_lines['distance_isolated_km'].min():.2f}–{unelec_lines['distance_isolated_km'].max():.2f} km")

print(f"\nNearest line type counts:")
print(unelec_lines["nearest_line_type"].value_counts().to_string())

print("\nBand distribution (No_tiene 2012 weighted):")
_bs = unelec_lines.groupby("dist_band", observed=True).agg(
    n_communities=("municipio_std", "count"),
    hh_no_tiene=("No_tiene", "sum"),
).assign(share=lambda d: (d["hh_no_tiene"] / d["hh_no_tiene"].sum()).round(3))
print(_bs.to_string())

In [ ]:
# SANITY CHECK — communities inside Riberalta and Cobija must be ~0 km from nearest MV line.
# Both cities have MV grid running through them. If minimum distance > 5 km,
# the CRS or geometry is wrong — stop and investigate.

print("SANITY CHECK — distance to nearest MV line for communities in Riberalta and Cobija")
print("Expected: communities physically inside the city ≈ 0 km")
print()

for muni, threshold_km in [("Riberalta", 2.0), ("Cobija", 2.0)]:
    sub = unelec_lines[unelec_lines["municipio_std"] == muni].sort_values("distance_km")
    cols = ["Comunidad", "distance_km", "distance_SIN_km", "distance_isolated_km", "nearest_line_type"]
    print(f"── {muni}  ({len(sub)} unelectrified communities, sorted by distance_km) ──")
    print(sub[cols].head(5).to_string(index=False))
    min_d = sub["distance_km"].min()
    if min_d > threshold_km:
        print(f"  !!! WARNING: minimum = {min_d:.2f} km  (expected < {threshold_km} km)")
        print(f"       Check CRS alignment between community points and MV line layer.")
    else:
        print(f"  OK: minimum distance = {min_d:.3f} km")
    print()

In [ ]:
# ============================================================
# Save Task 1 (lines) — full per-community distance table
# Merges plant-proxy distances from the old CSV for side-by-side reference.
# Output: task1_community_distances_lines.csv  (old file not overwritten)
# ============================================================

_plant_csv = OUTPUT_DIR / "task1_community_distances.csv"
if _plant_csv.exists():
    _old = pd.read_csv(_plant_csv, encoding="utf-8-sig")
    _plant = _old[["Municipio_std", "Comunidad", "distance_km"]].rename(
        columns={"distance_km": "distance_plant_km", "Municipio_std": "municipio_std"}
    )
    # Drop duplicates on join key — some community names repeat within a municipality
    _plant = _plant.drop_duplicates(subset=["municipio_std", "Comunidad"], keep="first")
    _working = unelec_lines.merge(_plant, on=["municipio_std", "Comunidad"], how="left")
    n_matched = _working["distance_plant_km"].notna().sum()
    print(f"Matched {n_matched}/{len(_working)} communities to plant-proxy distances from old CSV.")
    if n_matched < len(_working):
        print(f"  Note: {len(_working) - n_matched} communities unmatched.")
else:
    print(f"WARNING: {_plant_csv} not found — distance_plant_km will be absent.")
    _working = unelec_lines.copy()
    _working["distance_plant_km"] = float("nan")

task1_lines_out = _working[[
    "municipio_std", "cluster", "Municipio", "Comunidad",
    "No_tiene", "poblacion", "hh_2024_scaled",
    "distance_km", "nearest_line_type",
    "distance_SIN_km", "distance_isolated_km",
    "distance_plant_km", "dist_band",
]].copy()
task1_lines_out.columns = [
    "Municipio_std", "Cluster", "Municipio_raw", "Comunidad",
    "No_tiene_2012", "Poblacion_2012", "HH_2024_scaled",
    "distance_km", "nearest_line_type",
    "distance_SIN_km", "distance_isolated_km",
    "distance_plant_km", "Dist_band_lines",
]
task1_lines_out = task1_lines_out.sort_values(
    ["Cluster", "Municipio_std", "distance_km"]
).reset_index(drop=True)

out_path = OUTPUT_DIR / "task1_community_distances_lines.csv"
task1_lines_out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nSaved: {out_path}  ({task1_lines_out.shape[0]} rows × {task1_lines_out.shape[1]} cols)")
print("\nSample (5 rows):")
print(task1_lines_out[[
    "Municipio_std", "Cluster", "Comunidad",
    "distance_km", "nearest_line_type", "distance_SIN_km",
    "distance_isolated_km", "distance_plant_km",
]].head(5).to_string(index=False))

In [ ]:
# ============================================================
# Task 2 (lines) — Band-shift diff: plant proxy → real MV line distances
# ============================================================
# Uses hh_2024_scaled as the weight (same Source C scaling as before).
# Plant distances come from task1_community_distances.csv (old run).
# Line distances come from unelec_lines (computed in this session).

_plant_csv = OUTPUT_DIR / "task1_community_distances.csv"
if not _plant_csv.exists():
    raise FileNotFoundError(
        f"{_plant_csv} not found. Run the original notebook first to generate plant-proxy distances."
    )

_old_df = pd.read_csv(_plant_csv, encoding="utf-8-sig")
_old_df = _old_df.rename(columns={
    "Municipio_std": "municipio_std",
    "distance_km":   "distance_plant_km",
})
_old_df_dedup = _old_df[["municipio_std","Comunidad","distance_plant_km"]].drop_duplicates(
    subset=["municipio_std","Comunidad"], keep="first"
)

# Merge plant distances onto unelec_lines
_diff_base = unelec_lines.merge(_old_df_dedup, on=["municipio_std","Comunidad"], how="left")
n_merged = _diff_base["distance_plant_km"].notna().sum()
print(f"Merged plant distances: {n_merged}/{len(_diff_base)} communities matched.")
_diff_base = _diff_base.dropna(subset=["distance_plant_km","hh_2024_scaled"])

# Classify plant distances into bands
_diff_base["dist_band_plant"] = pd.cut(
    _diff_base["distance_plant_km"],
    bins=BAND_BINS, labels=BAND_LABELS, right=True, include_lowest=True,
)

# Per-cluster, per-band HH sums under each method
def _cluster_band_shares(df, band_col):
    grp = df.groupby(["cluster", band_col], observed=True)["hh_2024_scaled"].sum().reset_index()
    grp.columns = ["cluster", "dist_band", "hh"]
    tot = df.groupby("cluster")["hh_2024_scaled"].sum().rename("cluster_total")
    grp = grp.merge(tot, on="cluster")
    grp["share"] = grp["hh"] / grp["cluster_total"]
    return grp

_plant_shares = _cluster_band_shares(_diff_base, "dist_band_plant").rename(
    columns={"hh": "hh_plant", "share": "share_plant"}
)
_line_shares = _cluster_band_shares(_diff_base, "dist_band").rename(
    columns={"hh": "hh_lines", "share": "share_lines"}
)

# Convert Categorical dist_band to str before outer merge so fillna(0) works
_plant_shares["dist_band"] = _plant_shares["dist_band"].astype(str)
_line_shares["dist_band"]  = _line_shares["dist_band"].astype(str)

_diff = pd.merge(
    _plant_shares[["cluster","dist_band","hh_plant","share_plant"]],
    _line_shares [["cluster","dist_band","hh_lines","share_lines"]],
    on=["cluster","dist_band"],
    how="outer",
).fillna(0)

# Sort by cluster then by the canonical band order
_band_order = {b: i for i, b in enumerate(BAND_LABELS)}
_diff["_ord"] = _diff["dist_band"].map(_band_order)
_diff = _diff.sort_values(["cluster","_ord"]).drop(columns=["_ord"])

_diff["Δ_pp"] = ((_diff["share_lines"] - _diff["share_plant"]) * 100).round(1)

print()
print("=" * 82)
print("Task 2 (lines) — Band-shift diff: plant proxy → real MV line distances")
print("  share_plant  = HH share using old plant proxy distance")
print("  share_lines  = HH share using new MV line distance")
print("  Δ_pp         = (share_lines − share_plant) × 100  [percentage points]")
print("=" * 82)
_out = _diff.rename(columns={"cluster":"Cluster","dist_band":"Band"})
print(_out[["Cluster","Band","hh_plant","share_plant","hh_lines","share_lines","Δ_pp"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Per-cluster >50 km share change — flag if |Δ| > 5 pp
print()
print("─── Change in >50 km share per cluster (flagged if |Δ| > 5 pp) ───")
_b50 = _diff[_diff["dist_band"] == ">50 km"].set_index("cluster")
for cl in sorted(set(CLUSTER_MAP.values())):
    if cl in _b50.index:
        row = _b50.loc[cl]
        flag = "  *** FLAGGED" if abs(row["Δ_pp"]) > 5 else ""
        print(f"  {cl}:  plant={row['share_plant']*100:.1f}%  "
              f"lines={row['share_lines']*100:.1f}%  "
              f"Δ={row['Δ_pp']:+.1f} pp{flag}")
    else:
        print(f"  {cl}:  no communities in >50 km band")

In [ ]:
# ============================================================
# Task 3 (lines) — Cluster × distance band table (real MV line distances)
# Same columns as original task3_cluster_band.csv.
# Output: task3_cluster_band_lines.csv  (old file not overwritten)
# ============================================================

task3_lines = unelec_lines.groupby(["cluster", "dist_band"], observed=True).agg(
    HH_2024_scaled = ("hh_2024_scaled", "sum"),
    N_communities  = ("municipio_std",   "count"),
    Mean_dist_km   = ("distance_km",     "mean"),
    Median_dist_km = ("distance_km",     "median"),
).reset_index()

_cl_tot = unelec_lines.groupby("cluster")["hh_2024_scaled"].sum().rename("cluster_total_2024")
task3_lines = task3_lines.merge(_cl_tot, on="cluster")
task3_lines["Share_of_cluster"] = (task3_lines["HH_2024_scaled"] / task3_lines["cluster_total_2024"]).round(4)

task3_lines["HH_2024_scaled"] = task3_lines["HH_2024_scaled"].round(1)
task3_lines["Mean_dist_km"]   = task3_lines["Mean_dist_km"].round(2)
task3_lines["Median_dist_km"] = task3_lines["Median_dist_km"].round(2)

task3_lines = task3_lines.rename(columns={"dist_band": "Dist_band", "cluster": "Cluster"})
task3_lines = task3_lines[[
    "Cluster", "Dist_band", "HH_2024_scaled", "N_communities",
    "Mean_dist_km", "Median_dist_km", "Share_of_cluster",
]]

print("=" * 75)
print("Task 3 (lines) — Cluster × Distance band  (real MV line distances)")
print("=" * 75)
print(task3_lines.to_string(index=False))

print("\n─── Share of HH in >50 km band per cluster (line distances) ───")
_b50l = task3_lines[task3_lines["Dist_band"] == ">50 km"].set_index("Cluster")["Share_of_cluster"]
for cl in sorted(set(CLUSTER_MAP.values())):
    sh = _b50l.get(cl, 0.0)
    print(f"  {cl}: {sh * 100:.1f}%")

out_path = OUTPUT_DIR / "task3_cluster_band_lines.csv"
task3_lines.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nSaved: {out_path}")